# Contract

A contract combines a supplier tariff, a distributor tariff, government fees and taxes into a single object that calculates the full energy bill.

The easiest way to get started is to load a contract from a YAML config that references a preconfigured region and a registered supplier.

### 1. Setup indexes
Most supplier tariffs depend on an index. Register the required indexes before loading a contract.
See the [index documentation](./index.ipynb) for details.

In [1]:
from os import environ

from energy_cost.index import CachedEntsoeDayAheadIndex, Index
from helpers import MockEntsoeDayAheadIndex  # ty: ignore[unresolved-import]

# Index.register("spot", CachedEntsoeDayAheadIndex(country_code="BE", api_key=environ["ENTSOE_API_KEY"]))
Index.register("spot", MockEntsoeDayAheadIndex(country_code="BE"))

### 2. Registering a supplier
Suppliers are registered with their product tariffs. Contracts can then reference them by `supplier_key` and `product_key`:

In [2]:
from energy_cost import Supplier, Tariff

Supplier.register(
    "my_supplier",
    Supplier(
        products={
            "fixed": Tariff.from_yaml("../examples/tariffs/fixed.yml"),
            "dynamic": Tariff.from_yaml("../examples/tariffs/dynamic.yml"),
            "injection": Tariff.from_yaml("../examples/tariffs/injection.yml"),
        }
    ),
)

### 3. Loading a contract from YAML
A contract config uses `supplier_key` and `product_key` to look up the supplier, and `region`, `connection_type`, `customer_type` and `distributor_key` to resolve the distributor, fees and taxes:

In [3]:
from helpers import display_yaml  # ty: ignore[unresolved-import]

display_yaml("../examples/contracts/simple.yml")

```yaml
# A simple contract using reference keys to resolve everything.
# The supplier is looked up from the Supplier registry, and
# fees, taxes, distributor and timezone are resolved from the region.
supplier_key: my_supplier
product_key: fixed
region: be_flanders
connection_type: electricity
customer_type: residential
distributor_key: fluvius_imewo

```

In [4]:
import datetime as dt

import pandas as pd

from energy_cost import Contract, Meter
from energy_cost.meter import TimeseriesFrame

contract = Contract.from_yaml("../examples/contracts/simple.yml")

consumption = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0001,
        }
    ))
)

contract.apply(consumption)


timestamp    supplier        distributor              \
                            consumption  total    capacity consumption   
                                  total  total       total      levies   
0 2024-01-01 00:00:00+01:00       29.76  29.76    8.209764    0.399766   
1 2024-02-01 00:00:00+01:00       27.84  27.84    8.209764    0.373975   
2 2024-03-01 00:00:00+01:00        0.01   0.01    8.209764    0.000134   

                                                                               \
                                                       fixed            total   
  public_service_fee      total transmission data_management total      total   
0           8.305004  11.838260     3.133490            1.19  1.19  21.238025   
1           7.769197  11.074501     2.931329            1.19  1.19  20.474266   
2           0.002791   0.003978     0.001053            1.19  1.19   9.403742   

                 fees                                       total     taxes  
          consumption                            total      total     total  
  energy_contribution     excise      total      total      total     total  
0            0.573207  14.130048  14.703255  14.703255  69.643357  3.942077  
1            0.536226  13.218432  13.754658  13.754658  65.793060  3.724135  
2            0.000193   0.004748   0.004941   0.004941   9.983804  0.565121

### 4. Inline supplier tariffs
Instead of referencing a registered supplier, you can define the tariff directly in the contract YAML:

In [5]:
display_yaml("../examples/contracts/inline.yml")

```yaml
# A contract with the supplier tariff defined inline.
# Fees, taxes, distributor and timezone are resolved from the region.
supplier:
- start: 2025-01-01T00:00:00+01:00
  consumption:
    constant_cost: 90.0
region: be_flanders
connection_type: electricity
customer_type: residential
distributor_key: fluvius_imewo

```

In [6]:
contract = Contract.from_yaml("../examples/contracts/inline.yml")

consumption = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2025-01-01T00:00:00+01:00", "2025-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0001,
        }
    ))
)

contract.apply(consumption)


timestamp    supplier         distributor              \
                            consumption   total    capacity consumption   
                                  total   total       total      levies   
0 2025-01-01 00:00:00+01:00      26.784  26.784   11.092133    0.536394   
1 2025-02-01 00:00:00+01:00      24.192  24.192   11.092133    0.484485   
2 2025-03-01 00:00:00+01:00       0.009   0.009   11.092133    0.000180   

                                                                        \
                                                       fixed             
  public_service_fee      total transmission data_management     total   
0           9.668994  17.529563     7.324174        1.459167  1.459167   
1           8.733285  15.833153     6.615383        1.459167  1.459167   
2           0.003249   0.005890     0.002461        1.459167  1.459167   

                            fees                                       total  \
       total         consumption                            total      total   
       total energy_contribution     excise      total      total      total   
0  30.080862            0.573207  14.130048  14.703255  14.703255  75.862205   
1  28.384453            0.517736  12.762624  13.280360  13.280360  69.808221   
2  12.557190            0.000193   0.004748   0.004941   0.004941  13.325398   

      taxes  
      total  
      total  
0  4.294087  
1  3.951409  
2  0.754268

### 5. Fully inlined contract
When a region is not supported, or you need a special case, you can define all tariff components directly in the contract YAML — no registry lookups needed.

See the [tariff](./tariff.ipynb) and [tax](./tax.ipynb) notebooks for the full range of formula options.

In [7]:
display_yaml("../examples/contracts/inline_full.yml")

```yaml
# A fully inlined contract — all tariff components defined directly.
# Useful when the region is not supported or for handling special cases.
supplier:
- start: 2025-01-01T00:00:00+01:00
  consumption:
    constant_cost: 90.0
  injection:
    constant_cost: -15.0

distributor:
- start: 2025-01-01T00:00:00+01:00
  capacity:
    constant_cost: 50.0
  fixed:
    grid_fee:
      period: P1M
      constant_cost: 10.0

fees:
- start: 2025-01-01T00:00:00+01:00
  consumption:
    constant_cost: 20.0

taxes:
- start: 2025-01-01T00:00:00+01:00
  default: 0.06
  rates:
  - rate: 0.0
    columns:
    - ["*", "injection", "*"]

capacity_rule:
  measurement_period: PT15M
  billing_period: P1M

timezone: Europe/Brussels

```

In [8]:
contract = Contract.from_yaml("../examples/contracts/inline_full.yml")

consumption = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2025-01-01T00:00:00+01:00", "2025-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0001,
        }
    ))
)

contract.apply(consumption)


timestamp    supplier         distributor                 \
                            consumption   total    capacity    fixed         
                                  total   total       total grid_fee total   
0 2025-01-01 00:00:00+01:00      26.784  26.784      14.880     10.0  10.0   
1 2025-02-01 00:00:00+01:00      24.192  24.192      13.440     10.0  10.0   
2 2025-03-01 00:00:00+01:00       0.009   0.009       0.005     10.0  10.0   

                 fees            total    taxes  
    total consumption  total     total    total  
    total       total  total     total    total  
0  24.880       5.952  5.952  61.07296  3.45696  
1  23.440       5.376  5.376  56.18848  3.18048  
2  10.005       0.002  0.002  10.61696  0.60096

### 6. Resolution
By default the cost is calculated per month. Pass a different `resolution` for yearly or daily aggregation:

In [9]:
import isodate

contract = Contract.from_yaml("../examples/contracts/simple.yml")

consumption = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    ))
)

contract.apply(consumption, output_resolution=isodate.Duration(years=1))


timestamp    supplier                distributor  \
                            consumption  fixed   total    capacity   
                                  total  total   total       total   
0 2024-01-01 00:00:00+01:00      702.72    0.0  702.72   98.517173   
1 2025-01-01 00:00:00+01:00      630.72  120.0  750.72  133.105596   
2 2026-01-01 00:00:00+01:00      135.96  180.0  315.96  135.502454   

                                                                           \
  consumption                                                       fixed   
       levies public_service_fee       total transmission data_management   
0    9.439638         196.105260  279.535692    73.990794           14.28   
1   12.631219         227.689219  412.792925   172.472486           17.51   
2    1.626422          28.234133   59.240491    29.379936           17.85   

                                    fees                                      \
               total         consumption                               total   
   total       total energy_contribution      excise       total       total   
0  14.28  392.332865           13.535090  333.651456  347.186546  347.186546   
1  17.51  563.408521           13.498109  332.739840  346.237949  346.237949   
2  17.85  212.592945            2.182271   53.794840   55.977111   55.977111   

         total      taxes  
         total      total  
         total      total  
0  1528.773775  86.534365  
1  1759.988458  99.621988  
2   619.601860  35.071803

### 7. Time range
By default `start` and `end` match the data range. Override them to limit or extend the billing window — useful for the Flemish capacity tariff that needs 12 months of historical data:

In [10]:
from zoneinfo import ZoneInfo

consumption = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": [0.002] * 4 * 24 * 365 + [0.003] * 4 * 24 * 365 + [0.003] * 4 * 24 * 60 + [0.003],
        }
    ))
)

contract.apply(
    consumption,
    start=dt.datetime(2025, 1, 1, tzinfo=ZoneInfo("Europe/Brussels")),
    end=dt.datetime(2025, 12, 31, tzinfo=ZoneInfo("Europe/Brussels")),
)


timestamp    supplier               distributor  \
                             consumption fixed   total    capacity   
                                   total total   total       total   
0  2025-01-01 00:00:00+01:00      803.52  10.0  813.52   38.452728   
1  2025-02-01 00:00:00+01:00      725.76  10.0  735.76   39.931679   
2  2025-03-01 00:00:00+01:00      802.44  10.0  812.44   41.410630   
3  2025-04-01 00:00:00+02:00      777.60  10.0  787.60   42.889581   
4  2025-05-01 00:00:00+02:00      803.52  10.0  813.52   44.368532   
5  2025-06-01 00:00:00+02:00      777.60  10.0  787.60   45.847483   
6  2025-07-01 00:00:00+02:00      803.52  10.0  813.52   47.326434   
7  2025-08-01 00:00:00+02:00      803.52  10.0  813.52   48.805385   
8  2025-09-01 00:00:00+02:00      777.60  10.0  787.60   50.284336   
9  2025-10-01 00:00:00+02:00      804.60  10.0  814.60   51.763287   
10 2025-11-01 00:00:00+01:00      777.60  10.0  787.60   53.242238   
11 2025-12-01 00:00:00+01:00      803.52  10.0  813.52   53.242238   

                                                                            \
   consumption                                                       fixed   
        levies public_service_fee       total transmission data_management   
0    16.091827         290.069827  525.886877   219.725222        1.459167   
1    14.534554         261.998554  474.994598   198.461491        1.459167   
2    16.070198         289.679948  525.180040   219.429893        1.459167   
3    15.572736         280.712736  508.922784   212.637312        1.459167   
4    16.091827         290.069827  525.886877   219.725222        1.459167   
5    15.572736         280.712736  508.922784   212.637312        1.459167   
6    16.091827         290.069827  525.886877   219.725222        1.459167   
7    16.091827         290.069827  525.886877   219.725222        1.459167   
8    15.572736         280.712736  508.922784   212.637312        1.459167   
9    16.113456         290.459706  526.593714   220.020552        1.459167   
10   15.572736         280.712736  508.922784   212.637312        1.459167   
11   16.091827         290.069827  525.886877   219.725222        1.459167   

                                        fees                          \
                   total         consumption                           
       total       total energy_contribution      excise       total   
0   1.459167  565.798771           17.196221  406.114744  423.310965   
1   1.459167  516.385444           15.532070  366.813317  382.345388   
2   1.459167  568.049836           17.173108  405.568891  422.741999   
3   1.459167  553.271532           16.641504  393.014268  409.655772   
4   1.459167  571.714575           17.196221  406.114744  423.310965   
5   1.459167  556.229434           16.641504  393.014268  409.655772   
6   1.459167  574.672478           17.196221  406.114744  423.310965   
7   1.459167  576.151429           17.196221  406.114744  423.310965   
8   1.459167  560.666287           16.641504  393.014268  409.655772   
9   1.459167  579.816168           17.219334  406.660597  423.879931   
10  1.459167  563.624189           16.641504  393.014268  409.655772   
11  1.459167  580.588282           17.196221  406.114744  423.310965   

                      total       taxes  
         total        total       total  
         total        total       total  
0   423.310965  1910.787520  108.157784  
1   382.345388  1732.560281   98.069450  
2   422.741999  1911.425745  108.193910  
3   409.655772  1855.558942  105.031638  
4   423.310965  1917.058273  108.512732  
5   409.655772  1858.694319  105.209112  
6   423.310965  1920.193649  108.690207  
7   423.310965  1921.761337  108.778944  
8   409.655772  1863.397383  105.475324  
9   423.879931  1927.393865  109.097766  
10  409.655772  1866.532759  105.652798  
11  423.310965  1926.464401  109.045155

### 8. Dynamic pricing
A dynamic contract uses an index-based supplier tariff:

In [11]:
display_yaml("../examples/contracts/dynamic.yml")

```yaml
# A dynamic contract where the supplier tariff tracks an index.
supplier_key: my_supplier
product_key: dynamic
region: be_flanders
connection_type: electricity
customer_type: residential
distributor_key: fluvius_imewo

```

In [12]:
contract = Contract.from_yaml("../examples/contracts/dynamic.yml")

consumption = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0001,
        }
    ))
)

contract.apply(consumption)


timestamp distributor                                 \
                               capacity consumption                      
                                  total      levies public_service_fee   
0 2024-01-01 00:00:00+01:00    8.209764    0.399766           8.305004   
1 2024-02-01 00:00:00+01:00    8.209764    0.373975           7.769197   
2 2024-03-01 00:00:00+01:00    8.209764    0.000134           0.002791   

                                                            \
                                    fixed            total   
       total transmission data_management total      total   
0  11.838260     3.133490            1.19  1.19  21.238025   
1  11.074501     2.931329            1.19  1.19  20.474266   
2   0.003978     0.001053            1.19  1.19   9.403742   

                 fees                                       total     taxes  
          consumption                            total      total     total  
  energy_contribution     excise      total      total      total     total  
0            0.573207  14.130048  14.703255  14.703255  38.097757  2.156477  
1            0.536226  13.218432  13.754658  13.754658  36.282660  2.053735  
2            0.000193   0.004748   0.004941   0.004941   9.973204  0.564521

### 9. Injection
Provide injection as a separate `Meter` with `direction=PowerDirection.INJECTION`. The injection cost is subtracted from the consumption cost:

In [13]:
display_yaml("../examples/contracts/injection.yml")

```yaml
# A contract with both consumption and injection tariffs.
supplier_key: my_supplier
product_key: injection
region: be_flanders
connection_type: electricity
customer_type: residential
distributor_key: fluvius_imewo

```

In [14]:
from energy_cost import PowerDirection

contract = Contract.from_yaml("../examples/contracts/injection.yml")

consumption = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    ))
)

injection = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )),
    direction=PowerDirection.INJECTION,
)

contract.apply(consumption, injection=injection, output_resolution=isodate.Duration(years=1))


timestamp    supplier                          distributor  \
                            consumption   injection        total    capacity   
                                  total       total        total       total   
0 2024-01-01 00:00:00+01:00         NaN         NaN          NaN   98.517173   
1 2025-01-01 00:00:00+01:00  696.743108  596.822008  1293.565116  133.105596   
2 2026-01-01 00:00:00+01:00  112.906729   96.739742   209.646471  135.502454   

                                                                           \
  consumption                                                       fixed   
       levies public_service_fee       total transmission data_management   
0    9.439638         196.105260  279.535692    73.990794           14.28   
1   12.631219         227.689219  412.792925   172.472486           17.51   
2    1.626422          28.234133   59.240491    29.379936           17.85   

                                    fees                                      \
               total         consumption                               total   
   total       total energy_contribution      excise       total       total   
0  14.28  392.332865           13.535090  333.651456  347.186546  347.186546   
1  17.51  563.408521           13.498109  332.739840  346.237949  346.237949   
2  17.85  212.592945            2.182271   53.794840   55.977111   55.977111   

         total      taxes  
         total      total  
         total      total  
0   739.519411   0.000000  
1  2299.594960  96.383375  
2   501.105134  22.888607

### 10. Gas contracts
Gas contracts work the same way but use gas-specific regional data:

In [15]:
display_yaml("../examples/contracts/gas.yml")

```yaml
# A gas contract using the gas regional data.
supplier_key: my_supplier
product_key: fixed
region: be_flanders
connection_type: gas
customer_type: residential
distributor_key: fluvius_imewo

```

In [16]:
contract = Contract.from_yaml("../examples/contracts/gas.yml")

consumption = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    ))
)

contract.apply(consumption)


timestamp    supplier         distributor               \
                            consumption  total  consumption                
                                  total  total other_levies pension_levy   
0 2024-01-01 00:00:00+01:00       59.52  59.52     0.064341     0.071960   
1 2024-02-01 00:00:00+01:00       55.68  55.68     0.060190     0.067317   
2 2024-03-01 00:00:00+01:00        0.02   0.02     0.000022     0.000024   

                                                                               \
                                                                                
  proportional_distribution_fee public_service_obligation     total transport   
0                      4.562029                  0.282006  5.890992  0.910656   
1                      4.267705                  0.263812  5.510928  0.851904   
2                      0.001533                  0.000095  0.001979  0.000306   

                                                               \
            fixed                                       total   
  data_management fixed_distribution_fee     total      total   
0        1.096667               7.620410  8.717077  14.608069   
1        1.096667               7.128770  8.225437  13.736365   
2        1.096667               0.002561  1.099227   1.101207   

                 fees                                    total     taxes  
          consumption                         total      total     total  
  energy_contribution    excise     total     total      total     total  
0            0.593891  4.898496  5.492387  5.492387  84.397682  4.777227  
1            0.555575  4.582464  5.138039  5.138039  79.027668  4.473264  
2            0.000200  0.001646  0.001846  0.001846   1.190435  0.067383

### 11. Contract history
A `ContractHistory` tracks multiple sequential contracts and applies the correct one for each time segment:

In [17]:
display_yaml("../examples/contracts/history.yml")

```yaml
# A contract history tracking two sequential supplier contracts.
# Fees, taxes and distributor are resolved from the region.
- start: 2024-01-01T00:00:00+01:00
  end: 2025-06-01T00:00:00+02:00
  supplier_key: my_supplier
  product_key: fixed
  region: be_flanders
  connection_type: electricity
  customer_type: residential
  distributor_key: fluvius_imewo

- start: 2025-06-01T00:00:00+02:00
  supplier_key: my_supplier
  product_key: dynamic
  region: be_flanders
  connection_type: electricity
  customer_type: residential
  distributor_key: fluvius_imewo

```

In [18]:
from energy_cost import ContractHistory

history = ContractHistory.from_yaml("../examples/contracts/history.yml")

consumption = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    ))
)

history.apply(consumption)


timestamp distributor                                 \
                                capacity consumption                      
                                   total      levies public_service_fee   
0  2024-01-01 00:00:00+01:00    8.209764    0.799532          16.610008   
1  2024-02-01 00:00:00+01:00    8.209764    0.747949          15.538395   
2  2024-03-01 00:00:00+01:00    8.209764    0.798458          16.587683   
3  2024-04-01 00:00:00+02:00    8.209764    0.773741          16.074202   
4  2024-05-01 00:00:00+02:00    8.209764    0.799532          16.610008   
5  2024-06-01 00:00:00+02:00    8.209764    0.773741          16.074202   
6  2024-07-01 00:00:00+02:00    8.209764    0.799532          16.610008   
7  2024-08-01 00:00:00+02:00    8.209764    0.799532          16.610008   
8  2024-09-01 00:00:00+02:00    8.209764    0.773741          16.074202   
9  2024-10-01 00:00:00+02:00    8.209764    0.800607          16.632334   
10 2024-11-01 00:00:00+01:00    8.209764    0.773741          16.074202   
11 2024-12-01 00:00:00+01:00    8.209764    0.799532          16.610008   
12 2025-01-01 00:00:00+01:00   11.092133    1.072788          19.337988   
13 2025-02-01 00:00:00+01:00   11.092133    0.968970          17.466570   
14 2025-03-01 00:00:00+01:00   11.092133    1.071347          19.311997   
15 2025-04-01 00:00:00+02:00   11.092133    1.038182          18.714182   
16 2025-05-01 00:00:00+02:00   11.092133    1.072788          19.337988   
17 2025-06-01 00:00:00+02:00   11.092133    1.038182          18.714182   
18 2025-07-01 00:00:00+02:00   11.092133    1.072788          19.337988   
19 2025-08-01 00:00:00+02:00   11.092133    1.072788          19.337988   
20 2025-09-01 00:00:00+02:00   11.092133    1.038182          18.714182   
21 2025-10-01 00:00:00+02:00   11.092133    1.074230          19.363980   
22 2025-11-01 00:00:00+01:00   11.092133    1.038182          18.714182   
23 2025-12-01 00:00:00+01:00   11.092133    1.072788          19.337988   
24 2026-01-01 00:00:00+01:00   11.291871    0.854410          14.832265   
25 2026-02-01 00:00:00+01:00   11.291871    0.771725          13.396884   
26 2026-03-01 00:00:00+01:00   11.291871    0.000287           0.004984   

                                                                 \
                                     fixed                total   
        total transmission data_management     total      total   
0   23.676520     6.266980        1.190000  1.190000  33.076285   
1   22.149003     5.862659        1.190000  1.190000  31.548767   
2   23.644697     6.258556        1.190000  1.190000  33.044461   
3   22.912762     6.064819        1.190000  1.190000  32.312526   
4   23.676520     6.266980        1.190000  1.190000  33.076285   
5   22.912762     6.064819        1.190000  1.190000  32.312526   
6   23.676520     6.266980        1.190000  1.190000  33.076285   
7   23.676520     6.266980        1.190000  1.190000  33.076285   
8   22.912762     6.064819        1.190000  1.190000  32.312526   
9   23.708344     6.275403        1.190000  1.190000  33.108108   
10  22.912762     6.064819        1.190000  1.190000  32.312526   
11  23.676520     6.266980        1.190000  1.190000  33.076285   
12  35.059125    14.648348        1.459167  1.459167  47.610425   
13  31.666307    13.230766        1.459167  1.459167  44.217606   
14  35.012003    14.628660        1.459167  1.459167  47.563302   
15  33.928186    14.175821        1.459167  1.459167  46.479485   
16  35.059125    14.648348        1.459167  1.459167  47.610425   
17  33.928186    14.175821        1.459167  1.459167  46.479485   
18  35.059125    14.648348        1.459167  1.459167  47.610425   
19  35.059125    14.648348        1.459167  1.459167  47.610425   
20  33.928186    14.175821        1.459167  1.459167  46.479485   
21  35.106248    14.668037        1.459167  1.459167  47.657547   
22  33.928186    14.175821        1.459167  1.459167  46.479485   
23  35.059125    14.648348

### 12. Gaps in contract history
When a contract has an explicit `end` and the next contract starts later, the gap produces no cost:

In [19]:
display_yaml("../examples/contracts/history_with_gap.yml")

```yaml
# A contract history with a gap between contracts.
# The gap period (March–June 2025) produces no cost.
- start: 2024-01-01T00:00:00+01:00
  end: 2025-03-01T00:00:00+01:00
  supplier_key: my_supplier
  product_key: fixed
  region: be_flanders
  connection_type: electricity
  customer_type: residential
  distributor_key: fluvius_imewo

- start: 2025-06-01T00:00:00+02:00
  supplier_key: my_supplier
  product_key: dynamic
  region: be_flanders
  connection_type: electricity
  customer_type: residential
  distributor_key: fluvius_imewo

```

In [20]:
history = ContractHistory.from_yaml("../examples/contracts/history_with_gap.yml")

history.apply(consumption)


timestamp distributor                                 \
                                capacity consumption                      
                                   total      levies public_service_fee   
0  2024-01-01 00:00:00+01:00    8.209764    0.799532          16.610008   
1  2024-02-01 00:00:00+01:00    8.209764    0.747949          15.538395   
2  2024-03-01 00:00:00+01:00    8.209764    0.798458          16.587683   
3  2024-04-01 00:00:00+02:00    8.209764    0.773741          16.074202   
4  2024-05-01 00:00:00+02:00    8.209764    0.799532          16.610008   
5  2024-06-01 00:00:00+02:00    8.209764    0.773741          16.074202   
6  2024-07-01 00:00:00+02:00    8.209764    0.799532          16.610008   
7  2024-08-01 00:00:00+02:00    8.209764    0.799532          16.610008   
8  2024-09-01 00:00:00+02:00    8.209764    0.773741          16.074202   
9  2024-10-01 00:00:00+02:00    8.209764    0.800607          16.632334   
10 2024-11-01 00:00:00+01:00    8.209764    0.773741          16.074202   
11 2024-12-01 00:00:00+01:00    8.209764    0.799532          16.610008   
12 2025-01-01 00:00:00+01:00   11.092133    1.072788          19.337988   
13 2025-02-01 00:00:00+01:00   11.092133    0.968970          17.466570   
14 2025-06-01 00:00:00+02:00   11.092133    1.038182          18.714182   
15 2025-07-01 00:00:00+02:00   11.092133    1.072788          19.337988   
16 2025-08-01 00:00:00+02:00   11.092133    1.072788          19.337988   
17 2025-09-01 00:00:00+02:00   11.092133    1.038182          18.714182   
18 2025-10-01 00:00:00+02:00   11.092133    1.074230          19.363980   
19 2025-11-01 00:00:00+01:00   11.092133    1.038182          18.714182   
20 2025-12-01 00:00:00+01:00   11.092133    1.072788          19.337988   
21 2026-01-01 00:00:00+01:00   11.291871    0.854410          14.832265   
22 2026-02-01 00:00:00+01:00   11.291871    0.771725          13.396884   
23 2026-03-01 00:00:00+01:00   11.291871    0.000287           0.004984   

                                                                 \
                                     fixed                total   
        total transmission data_management     total      total   
0   23.676520     6.266980        1.190000  1.190000  33.076285   
1   22.149003     5.862659        1.190000  1.190000  31.548767   
2   23.644697     6.258556        1.190000  1.190000  33.044461   
3   22.912762     6.064819        1.190000  1.190000  32.312526   
4   23.676520     6.266980        1.190000  1.190000  33.076285   
5   22.912762     6.064819        1.190000  1.190000  32.312526   
6   23.676520     6.266980        1.190000  1.190000  33.076285   
7   23.676520     6.266980        1.190000  1.190000  33.076285   
8   22.912762     6.064819        1.190000  1.190000  32.312526   
9   23.708344     6.275403        1.190000  1.190000  33.108108   
10  22.912762     6.064819        1.190000  1.190000  32.312526   
11  23.676520     6.266980        1.190000  1.190000  33.076285   
12  35.059125    14.648348        1.459167  1.459167  47.610425   
13  31.666307    13.230766        1.459167  1.459167  44.217606   
14  33.928186    14.175821        1.459167  1.459167  46.479485   
15  35.059125    14.648348        1.459167  1.459167  47.610425   
16  35.059125    14.648348        1.459167  1.459167  47.610425   
17  33.928186    14.175821        1.459167  1.459167  46.479485   
18  35.106248    14.668037        1.459167  1.459167  47.657547   
19  33.928186    14.175821        1.459167  1.459167  46.479485   
20  35.059125    14.648348        1.459167  1.459167  47.610425   
21  31.120865    15.434191        1.487500  1.487500  43.900236   
22  28.109169    13.940559        1.487500  1.487500  40.888540   
23   0.010457     0.005186        1.487500  1.487500  12.789828   

                  fees                                     supplier        \
           consumption                            total consumption fixed   
   energy_contribution     ex

### 13. Building a contract in code
For full control you can construct a `Contract` directly with custom tariffs and taxes.
See the [tariff](./tariff.ipynb) and [tax](./tax.ipynb) notebooks for details on defining these.

In [21]:
from energy_cost.data import CustomerType
from energy_cost.data.be.flanders.electricity import data

contract = Contract(
    supplier=Tariff.from_yaml("../examples/tariffs/dynamic.yml"),
    distributor=data.distributors["fluvius_imewo"],
    fees=data.fees[CustomerType.RESIDENTIAL],
    taxes=data.taxes,
    timezone=ZoneInfo("Europe/Brussels"),
    capacity_rule=data.capacity_rule
)

consumption = Meter(
    measurements=TimeseriesFrame(pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0001,
        }
    ))
)

contract.apply(consumption)


timestamp distributor                                 \
                               capacity consumption                      
                                  total      levies public_service_fee   
0 2024-01-01 00:00:00+01:00    8.209764    0.399766           8.305004   
1 2024-02-01 00:00:00+01:00    8.209764    0.373975           7.769197   
2 2024-03-01 00:00:00+01:00    8.209764    0.000134           0.002791   

                                                            \
                                    fixed            total   
       total transmission data_management total      total   
0  11.838260     3.133490            1.19  1.19  21.238025   
1  11.074501     2.931329            1.19  1.19  20.474266   
2   0.003978     0.001053            1.19  1.19   9.403742   

                 fees                                       total     taxes  
          consumption                            total      total     total  
  energy_contribution     excise      total      total      total     total  
0            0.573207  14.130048  14.703255  14.703255  38.097757  2.156477  
1            0.536226  13.218432  13.754658  13.754658  36.282660  2.053735  
2            0.000193   0.004748   0.004941   0.004941   9.973204  0.564521